# Flux Dev Photo Generator — Colab T4

**Портретные фото Instagram-качества** на базе Flux Dev fp8 + IPAdapter FaceID + перенос стиля.

| Воркфлоу | Вход | Выход |
|----------|------|-------|
| `photo_flux_instagram` | Референсные фото лица + текст | Портреты с консистентным FaceID |
| `photo_flux_img2img` | Референсное изображение + текст | Перенос стиля/композиции |

**Запускайте ячейки 1-6 по порядку**, затем откройте ссылку на туннель.

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Уже клонировано"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Кастомные ноды
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git 2>/dev/null || echo "Уже клонировано"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Уже клонировано"

!pip install insightface onnxruntime -q
!pip install -r ComfyUI_IPAdapter_plus/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

print("\nУстановлено!")

In [ ]:
#@title 3. Скачивание моделей (~18 ГБ)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/checkpoints", exist_ok=True)
os.makedirs(f"{M}/clip_vision", exist_ok=True)
os.makedirs(f"{M}/ipadapter", exist_ok=True)

files = {
    # Flux Dev fp8 (~11.5 ГБ)
    f"{M}/checkpoints/flux1-dev-fp8.safetensors":
        "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
    # CLIP Vision H14 (~3.9 ГБ)
    f"{M}/clip_vision/model.safetensors":
        "https://huggingface.co/laion/CLIP-ViT-H-14-laion2B-s32B-b79K/resolve/main/open_clip_pytorch_model.safetensors",
    # IPAdapter FaceID PlusV2 SDXL (~0.9 ГБ)
    f"{M}/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin":
        "https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin",
    # IPAdapter Flux (перенос стиля) (~1.5 ГБ)
    f"{M}/ipadapter/ip-adapter_flux.safetensors":
        "https://huggingface.co/InstantX/FLUX.1-dev-IP-Adapter/resolve/main/ip-adapter.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nВсе модели готовы!")
!du -sh {M}/checkpoints/* {M}/clip_vision/* {M}/ipadapter/*

In [ ]:
#@title 4. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

# Скачиваем Colab-версию (с исправленными путями)
!wget -q -O "{WF_DIR}/photo_flux_instagram.json" "{RAW}/photo_flux_instagram_colab.json"
print("Скачано: photo_flux_instagram (пути для Colab)")

# Скачиваем воркфлоу img2img
!wget -q -O "{WF_DIR}/photo_flux_img2img.json" "{RAW}/photo_flux_img2img.json"
print("Скачано: photo_flux_img2img")

print(f"\n2 воркфлоу сохранены в {WF_DIR}")

In [ ]:
#@title 5. Загрузка референсных фото
#@markdown Загрузите референсные фото лица. Они будут доступны в нодах LoadImage.
#@markdown В воркфлоу 3 слота для папок: **portrait**, **hall**, **street**.

from google.colab import files
import os

# Создаём папки для референсов, соответствующие структуре воркфлоу
INPUT = "/content/ComfyUI/input"
for folder in ["refs/portrait", "refs/hall", "refs/street"]:
    os.makedirs(f"{INPUT}/{folder}", exist_ok=True)

target = "refs/portrait" #@param ["refs/portrait", "refs/hall", "refs/street"]

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(INPUT, target, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Сохранено: {path}")

print(f"\nФайлы в {target}: {os.listdir(os.path.join(INPUT, target))}")

In [ ]:
#@title 6. Запуск ComfyUI
import subprocess, time, re

# Установка cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Запуск ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("Запуск ComfyUI... (30 сек)")
time.sleep(30)

# Cloudflare туннель
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI готов! Открывайте: {url}")
    print(f"{'='*60}")
    print("\nЗагрузите воркфлоу: Меню -> Load -> photo_flux_instagram")
else:
    print("Ссылка на туннель не найдена. ComfyUI работает на порту 8188.")

In [ ]:
#@title 7. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/flux"
os.makedirs(dst, exist_ok=True)

images = glob.glob(f"{src}/*.png") + glob.glob(f"{src}/*.jpg")
if not images:
    print("Результатов пока нет. Сначала сгенерируйте изображения!")
else:
    for img in images:
        shutil.copy2(img, dst)
        print(f"Скопировано: {os.path.basename(img)}")
    print(f"\n{len(images)} файлов сохранено в {dst}")

---
## Инструкция по использованию

### Воркфлоу 1: `photo_flux_instagram` — FaceID портреты
1. Откройте ссылку на туннель
2. Загрузите воркфлоу **photo_flux_instagram**
3. Воркфлоу загружает референсные фото лица из 3 папок (portrait/hall/street)
4. Используйте **ImageMaskSwitch** для выбора папки: 1=portrait, 2=hall, 3=street
5. Отредактируйте промпт и нажмите **Queue Prompt**

### Воркфлоу 2: `photo_flux_img2img` — Перенос стиля
1. Загрузите воркфлоу **photo_flux_img2img**
2. Загрузите референсное изображение (источник стиля/композиции)
3. Отредактируйте промпт, описав желаемый результат
4. IPAdapter переносит стиль, а промпт направляет содержание
5. Настройте `denoise` в KSampler (0.65 = умеренные изменения, 0.9 = сильные изменения)

### Использование LoRA
1. Запустите ячейку 8 для скачивания вашей LoRA
2. В ComfyUI добавьте ноду **LoraLoader** между CheckpointLoader и KSampler
3. Выберите файл вашей LoRA
4. Используйте триггер-слово в промпте (например, "a photo of ohwx person")
5. Сила LoRA 0.7-1.0 лучше всего подходит для лиц

### Пути к папкам (уже настроены для Colab)
- `/content/ComfyUI/input/refs/portrait`
- `/content/ComfyUI/input/refs/hall`
- `/content/ComfyUI/input/refs/street`

### Результат
- Разрешение: 864x1080 (портрет)
- Префикс: `instagram_flux` / `flux_img2img`
- Файлы сохраняются в `/content/ComfyUI/output/`

---
## Инструкция

1. Откройте ссылку на туннель
2. Загрузите воркфлоу **photo_flux_instagram**
3. Воркфлоу загружает референсные фото лица из 3 папок (portrait/hall/street)
4. Используйте **ImageMaskSwitch** для выбора папки: 1=portrait, 2=hall, 3=street
5. Отредактируйте промпт и нажмите **Queue Prompt**

### Пути к папкам в Colab
Обновите ноды `LoadImagesFromFolderKJ`, если пути отличаются:
- `/content/ComfyUI/input/refs/portrait`
- `/content/ComfyUI/input/refs/hall`
- `/content/ComfyUI/input/refs/street`

### Результат
- Разрешение: 864x1080 (портрет)
- Префикс: `instagram_flux`
- Файлы сохраняются в `/content/ComfyUI/output/`